# Vendor entity linking on Canadian federal contracts: full run

**Before you start (Kaggle):** Settings > Internet ON. For the Neo4j part, add three secrets
(Add-ons > Secrets): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` (from your free Neo4j Aura instance).

Steps: download data > slice > link > label > evaluate > entities > Neo4j > queries.

In [ ]:
# 1. Get the code (or upload the repo zip as a Kaggle dataset and copy it here)
!git clone -q https://github.com/Mathindata/canada-contracts-entity-linking.git
%cd canada-contracts-entity-linking
!pip install -q rapidfuzz neo4j
import os, sys, json
sys.path.insert(0, "src"); os.environ["PYTHONPATH"] = "src"
import pandas as pd
from IPython.display import Image, display

In [ ]:
# 2. Download the open data (~655 MB, a few minutes)
import urllib.request
URL = "https://open.canada.ca/data/dataset/d8f85d91-7dec-4fd1-8055-483b77225d8b/resource/fac950c0-00d5-4ec1-a4d3-9cbebf98a305/download/contracts.csv"
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/contracts.csv"):
    urllib.request.urlretrieve(URL, "data/raw/contracts.csv")
print(round(os.path.getsize("data/raw/contracts.csv") / 1e6), "MB")
head = pd.read_csv("data/raw/contracts.csv", nrows=5, dtype=str, encoding="utf-8-sig")
print(list(head.columns))
head[["vendor_name", "vendor_postal_code", "procurement_id", "instrument_type", "contract_date"]]

## Choose the slice
Edit `config.json`: `departments` (empty = the `top_n_departments` with most rows), `year_min`, `year_max`.
Aim for roughly 5,000 to 50,000 unique vendor strings. Check `results/slice_summary.json` after `prepare`.

In [ ]:
# 3. Slice and build unique vendor strings
!python -m cel prepare
json.load(open("results/slice_summary.json"))

In [ ]:
# 4. Blocking, comparisons, Fellegi-Sunter training (no labels), scoring
!python -m cel link
s = json.load(open("results/linking_summary.json")); s.pop("training_sessions"); print(json.dumps(s, indent=2))
pd.read_csv("results/match_weights.csv").round(3)

## Labels
`label-sheet` writes a stratified sample of pairs (scores hidden). Rules: `labels/README.md`.

**Kaggle wipes files when the session stops.** Either label here and download `labels/labeling_sheet.csv`
right after, or download the sheet, label on your laptop (`python -m cel label` or a spreadsheet), then upload it back.

In [ ]:
# 5. Write the labeling sheet (refuses to overwrite a sheet that already has labels)
!python -m cel label-sheet
pd.read_csv("labels/labeling_sheet.csv").head()

In [ ]:
# 6. Label here, pair by pair (y / n / u / q). Saves after every answer; re-run to resume.
from cel.cli import step_label_interactive
step_label_interactive("labels")

In [ ]:
# 7. Evaluate: weighted precision/recall with bootstrap CIs, PR curve, calibration
!python -m cel evaluate
display(Image("results/pr_curve.png")); display(Image("results/reliability.png"))

In [ ]:
# 8. Vendor entities and the chaining audit, then the graph tables
!python -m cel cluster
!python -m cel export-graph
print(json.load(open("results/cluster_summary.json")))
pd.read_csv("results/cluster_audit.csv").head(10)

## Neo4j (v2)
Create a free instance at Neo4j Aura, save its credentials as Kaggle secrets, then load and query.

In [ ]:
# 9. Credentials from Kaggle secrets (never paste the password into the notebook)
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    os.environ[k] = sec.get_secret(k)

In [ ]:
# 10. Load the graph (--wipe clears the database first) and run queries/*.cypher
!python -m cel load-neo4j --wipe
!python -m cel run-queries --min-conf 0.5

In [ ]:
# 11. Bundle results and labels for download / commit
!zip -qr outputs.zip results labels
print("download outputs.zip from the Output panel")